# Определение автоматизированного трафика

Задача — дать каждой `cookie_id` из `test.csv` score от 0 до 1: чем выше score, тем вероятнее автоматизированный сбор данных. Основная метрика организаторов — максимальный Precision при Recall ≥ 0.7. Проверка перебирает порог сама; сохранять его не нужно.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# Запуск возможен из этой папки или из её родительского каталога.
here = Path.cwd().resolve()
ROOT = here if (here / 'solution.ipynb').is_file() and (here / 'data/train.csv').is_file() else here / 'avito_antibot_final'
assert (ROOT / 'solution.ipynb').is_file() and (ROOT / 'data/train.csv').is_file()


## Данные и признаки

В `data/` лежат три неизменённых файла исходного архива: `train.csv`, `test.csv` и `events.csv.gz`. Для каждой cookie берутся события только внутри её суточного окна: `window_start_ts <= event_ts < window_end_ts`. Точные дубли удаляются, события сортируются по времени.

Строятся 94 числовых признака: типы действий и их доли, разнообразие просмотров и поиска, интервалы и концентрация активности, курсор, общие признаки платформы и User-Agent, возраст cookie, а также ритм по минутам и переходы между действиями. Неопределённые статистики заполняются константой `-1`. Каждая cookie обрабатывается независимо; `target` в функции признаков не передаётся.

In [2]:
train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])
assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert 'target' in train and 'target' not in test
assert train.target.notna().all() and set(train.target.unique()) == {0, 1}
assert train.window_start_ts.lt(train.window_end_ts).all()
assert test.window_start_ts.lt(test.window_end_ts).all()


In [3]:
EVENTS = ['item_view','search_results_view','photo_swipe','favorite_add','seller_page_view','contact_phone_show','login','contact_chat_open','contact_message_sent','captcha_shown']

def in_window(events, meta):
    # В расчёт передаются только метаданные окна, без target.
    m = meta[['cookie_id','cookie_created_at','window_start_ts','window_end_ts']]
    assert m.cookie_id.is_unique
    e = events.merge(m, on='cookie_id', validate='many_to_one')
    return e[e.event_ts.ge(e.window_start_ts) & e.event_ts.lt(e.window_end_ts)].copy()

def features(events, meta):
    e = in_window(events, meta).drop_duplicates().sort_values(['cookie_id','event_ts'])
    g = e.groupby('cookie_id', sort=False)
    f = pd.DataFrame(index=pd.Index(meta.cookie_id, name='cookie_id'))
    f['n_events'] = g.size()
    for name in EVENTS:
        counts = e.loc[e.event_name.eq(name)].groupby('cookie_id').size().reindex(f.index, fill_value=0)
        f['count_'+name] = counts
        f['share_'+name] = counts / f.n_events
    for col in ['item_id','item_category','item_location','search_query','seller_type']:
        f[col+'_unique'] = g[col].nunique()
        f[col+'_unique_ratio'] = g[col].nunique() / g[col].count().replace(0,np.nan)
    for stat in ['mean','std','max','median','nunique']:
        f['search_page_'+stat] = getattr(g.search_page,stat)()
    first, last = g.event_ts.min(), g.event_ts.max()
    f['activity_seconds'] = (last-first).dt.total_seconds()
    f['events_per_second'] = f.n_events/(f.activity_seconds+1)
    e['gap'] = g.event_ts.diff().dt.total_seconds()
    gaps = e.groupby('cookie_id').gap
    for stat in ['mean','std','min','max','median']:
        f['gap_'+stat] = getattr(gaps,stat)()
    f['gap_cv'] = f.gap_std/(f.gap_mean+1e-6)
    for key, mask in [('short',e.gap.le(1)),('long',e.gap.gt(300)),('zero',e.gap.eq(0))]:
        f['gap_'+key+'_share'] = mask.groupby(e.cookie_id).sum()/(f.n_events-1).clip(lower=1)
    e['hour'] = ((e.event_ts-e.window_start_ts).dt.total_seconds()//3600).astype(int)
    h = e.groupby(['cookie_id','hour']).size()
    f['active_hours'] = h.groupby(level=0).size()
    f['max_hour_share'] = h.groupby(level=0).max()/f.n_events
    hp = h / h.groupby(level=0).transform('sum')
    f['hour_entropy'] = (-(hp*np.log(hp))).groupby(level=0).sum()
    p = e.dropna(subset=['pointer_x','pointer_y']).copy()
    pg = p.groupby('cookie_id')
    f['pointer_share'] = pg.size()/f.n_events
    for col in ['pointer_x','pointer_y']:
        for stat in ['std','nunique']:
            f[col+'_'+stat] = getattr(pg[col],stat)()
    p['step'] = np.hypot(pg.pointer_x.diff(),pg.pointer_y.diff())
    for stat in ['mean','std','median']:
        f['pointer_step_'+stat] = getattr(p.groupby('cookie_id').step,stat)()
    f['pointer_repeat_share'] = p.step.eq(0).groupby(p.cookie_id).sum()/(pg.size()-1).clip(lower=1)
    e['platform_norm'] = e.platform.str.lower().replace({'desktop':'web','iphone':'ios'})
    f['platform_unique'] = e.groupby('cookie_id').platform_norm.nunique()
    for name in ['web','android','ios']:
        f['platform_'+name+'_share'] = e.platform_norm.eq(name).groupby(e.cookie_id).mean()
    ua = e.user_agent.fillna('').str.lower()
    for name,pattern in {'automation':'headless|selenium|python|curl|wget|bot|scrapy', 'mobile':'mobile|android|iphone', 'app':'^avito/', 'firefox':'firefox', 'chrome':'chrome', 'yandex':'yabrowser'}.items():
        f['ua_'+name+'_share'] = ua.str.contains(pattern,regex=True).groupby(e.cookie_id).mean()
    m = meta.set_index('cookie_id')
    f['cookie_age_days'] = (m.window_end_ts-m.cookie_created_at).dt.total_seconds()/86400
    assert not f.cookie_age_days.lt(0).any()
    f['no_events'] = f.n_events.isna().astype(int)
    # Неопределённые статистики получают постоянное значение.
    return f.replace([np.inf,-np.inf],np.nan).fillna(-1).astype(float)

def extra_features(events, meta):
    """Независимые агрегаты событий каждой cookie строго внутри её окна."""
    e = in_window(events, meta[['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']])
    e = e.drop_duplicates().sort_values(['cookie_id', 'event_ts'])
    g = e.groupby('cookie_id', sort=False)
    idx = pd.Index(meta.cookie_id, name='cookie_id')
    t = pd.DataFrame(index=idx)
    gap = g.event_ts.diff().dt.total_seconds()
    gg = gap.groupby(e.cookie_id)
    for q in [0.1, 0.25, 0.75, 0.9]:
        t[f'new_gap_q{int(q * 100)}'] = gg.quantile(q)
    t['new_gap_iqr'] = t.new_gap_q75 - t.new_gap_q25
    denom = (g.size() - 1).clip(lower=1)
    for name, mask in [
        ('1_to_5s', gap.gt(1) & gap.le(5)),
        ('5_to_60s', gap.gt(5) & gap.le(60)),
        ('over_15min', gap.gt(900)),
    ]:
        t[f'new_gap_{name}_share'] = mask.groupby(e.cookie_id).sum() / denom
    minute = ((e.event_ts - e.window_start_ts).dt.total_seconds() // 60).astype(int)
    per_minute = e.groupby([e.cookie_id, minute]).size()
    t['new_active_minutes'] = per_minute.groupby(level=0).size()
    t['new_max_events_per_minute'] = per_minute.groupby(level=0).max()
    t['new_top_minute_share'] = per_minute.groupby(level=0).max() / g.size()
    t['new_median_events_per_minute'] = per_minute.groupby(level=0).median()

    previous_action = g.event_name.shift()
    pairs = [
        ('item_to_item', 'item_view', 'item_view'),
        ('search_to_item', 'search_results_view', 'item_view'),
        ('item_to_search', 'item_view', 'search_results_view'),
        ('search_to_search', 'search_results_view', 'search_results_view'),
        ('item_to_photo', 'item_view', 'photo_swipe'),
    ]
    for name, before, after in pairs:
        flag = previous_action.eq(before) & e.event_name.eq(after)
        t[f'new_transition_{name}_share'] = flag.groupby(e.cookie_id).sum() / denom
    item_views = e[e.event_name.eq('item_view') & e.item_id.notna()]
    iv = item_views.groupby('cookie_id').item_id
    t['new_item_view_revisit_share'] = 1 - iv.nunique() / iv.size()
    item_previous = g.item_id.shift()
    consecutive_item = e.item_id.notna() & e.item_id.eq(item_previous)
    t['new_same_item_adjacent_share'] = consecutive_item.groupby(e.cookie_id).sum() / denom
    for col in ['item_category', 'item_location', 'search_query']:
        nonmissing = e[e[col].notna()].groupby('cookie_id')[col]
        top_count = nonmissing.value_counts().groupby(level=0).max()
        t[f'new_{col}_top_share'] = top_count / nonmissing.size()
    search = e[e.search_page.notna()]
    sg = search.groupby('cookie_id').search_page
    t['new_deep_search_share'] = search.search_page.gt(5).groupby(search.cookie_id).sum() / sg.size()
    # Для пропусков используется та же фиксированная константа.
    return t.replace([np.inf, -np.inf], np.nan).fillna(-1).astype(float)

In [4]:
metadata = pd.concat([train.drop(columns='target'), test], ignore_index=True)
base = features(events, metadata)
extra = extra_features(events, metadata)
assert base.shape == (len(metadata), 71) and extra.shape == (len(metadata), 23)
assert base.index.equals(extra.index)
all_features = pd.concat([base, extra], axis=1)
assert np.isfinite(all_features.to_numpy()).all()
X_train = all_features.loc[train.cookie_id]
X_test = all_features.loc[test.cookie_id]
assert X_train.columns.equals(X_test.columns)


## Модель

CatBoost обучается на всех размеченных cookie. Параметры не менялись после временной валидации: 500 итераций, глубина 6, шаг 0.05, фиксированный seed 0. Используется CPU; внешние API не вызываются.

In [5]:
model = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                           loss_function='Logloss', random_seed=0,
                           thread_count=4, verbose=False, allow_writing_files=False)
model.fit(X_train, train.target.to_numpy())
score = model.predict_proba(X_test)[:, 1]


## Временная валидация и результаты

Использовались два разбиения: обучение 6–16 апреля 2026 года с проверкой 17–19 апреля; обучение 6–13 апреля с проверкой 14–16 апреля. Официальная функция `precision_at_recall` дала **0.751678** и **0.713542**. После обучения на всём train файл этой версии получил на Stepik **0.79361**. Это результат проверки, а не локальная оценка на новых данных.

## Итоговые предсказания

Сохраняются ровно два столбца: `cookie_id` из `test.csv` и score модели. Каждая cookie встречается один раз.

In [6]:
submission = pd.DataFrame({'cookie_id': test.cookie_id, 'score': score})
assert list(submission.columns) == ['cookie_id', 'score']
assert submission.cookie_id.is_unique and submission.cookie_id.equals(test.cookie_id)
assert not submission.isna().any().any() and submission.score.between(0, 1).all()
output = ROOT / 'submission.csv'
submission.to_csv(output, index=False)
print(f'Готово: {output.name}, {len(submission)} строк')


Готово: submission.csv, 4909 строк
